In [1]:
import os, sys
_here = os.path.abspath(os.getcwd())
_root = _here if os.path.isdir(os.path.join(_here, 'pipeline')) else os.path.abspath(os.path.join(_here, os.pardir))
if _root not in sys.path:
    sys.path.insert(0, _root)
os.chdir(_root)
print('project root:', _root)

project root: /home/temari/god please no diploma/restore_punct


## Run config

In [ ]:
from pipeline.env import DATA_DIR, MODEL_DIR
from pipeline.config import RunConfig

LORUGEC_UPSAMPLE = 13
lorugec_files = [f"{DATA_DIR}/train_lorugec.json"] * LORUGEC_UPSAMPLE

cfg = RunConfig(
    name          = "04c_finetune_no_replay",
    architecture  = "bert+crf",
    loss          = "crf",
    train_files   = [
        f"{DATA_DIR}/train_gera.json",
        *lorugec_files,
    ],
    val_files     = [f"{DATA_DIR}/val_gera.json"],
    replay_files  = [],
    init_from     = f"{MODEL_DIR}/03_crf_transitions_mined",
    num_epochs    = 2,
    learning_rate = 5e-6,
    crf_init_transitions = False,
    tags          = {"stage": "4-gera-lorugec", "strategy": "finetune-no-replay"},
)


print(cfg)

RunConfig(name='04c_finetune_no_replay', architecture='bert+crf', loss='crf', train_files=['/home/temari/god please no diploma/restore_punct/data/train_gera.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', '/home/temari/god please no diploma/restore_punct/data/train_lorugec.json', 

## Train

In [3]:
from pipeline.training import train_run
model = train_run(cfg)

[04c_finetune_no_replay] architecture=bert+crf loss=crf epochs=2 lr=5e-06
[04c_finetune_no_replay] warm-starting from /home/temari/god please no diploma/restore_punct/models/03_crf_transitions_mined


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Map:   0%|          | 0/8747 [00:00<?, ? examples/s]

/home/temari/anaconda3/envs/neural/lib/python3.13/site-packages/torchcrf/__init__.py:249: UserWarning: where received a uint8 condition tensor. This behavior is deprecated and will be removed in a future version of PyTorch. Use a boolean condition instead. (Triggered internally at /pytorch/aten/src/ATen/native/TensorCompare.cpp:614.)
  score = torch.where(mask[i].unsqueeze(1), next_score, score)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.067865,1.978649,0.965317,0.966879,0.965401,0.966879
2,0.839613,1.985587,0.966024,0.967742,0.966266,0.967742


Model saved -> /home/temari/god please no diploma/restore_punct/models/04c_finetune_no_replay


## Benchmark + save results

In [4]:
from pipeline.evaluation import evaluate_and_save
reports = evaluate_and_save(cfg)

for test_name, rep in reports.items():
    wa = rep.get('weighted avg', {})
    ma = rep.get('macro avg', {})
    print(f"{test_name:>14s} | W-F1={wa.get('f1-score', 0):.4f}  M-F1={ma.get('f1-score', 0):.4f}  P={wa.get('precision', 0):.4f}  R={wa.get('recall', 0):.4f}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[04c_finetune_no_replay] evaluating on General_Test (n=569)


Evaluating:   0%|          | 0/143 [00:00<?, ?it/s]

[04c_finetune_no_replay] evaluating on GERA_Test (n=1259)


Evaluating:   0%|          | 0/315 [00:00<?, ?it/s]

[04c_finetune_no_replay] evaluating on LORuGEC_Test (n=603)


Evaluating:   0%|          | 0/151 [00:00<?, ?it/s]

Updated /home/temari/god please no diploma/restore_punct/results/models_db.json (entry: 04c_finetune_no_replay)
Wrote /home/temari/god please no diploma/restore_punct/results/04c_finetune_no_replay.json
Wrote /home/temari/god please no diploma/restore_punct/results/04c_finetune_no_replay.xlsx
  General_Test | W-F1=0.9485  M-F1=0.4218  P=0.9480  R=0.9511
     GERA_Test | W-F1=0.9673  M-F1=0.4877  P=0.9675  R=0.9686
  LORuGEC_Test | W-F1=0.9750  M-F1=0.4861  P=0.9749  R=0.9758


## Example run

In [5]:
from pipeline.inference import load_for_inference, restore_punctuation

m, tok = load_for_inference(cfg)
for t in [
    "Мама мыла раму а папа читал газету",
    "Однако мы решили не идти в кино потому что пошел дождь",
    "Он сказал Привет как дела",
    "Я говорю ей Что думаешь А она мне Да ничего отстань уже",
    "После многих страданий А А Пушкин всё-таки написал свои стишки",
]:
    print(f"Input:    {t}")
    print(f"Restored: {restore_punctuation(m, tok, t)}\n")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Input:    Мама мыла раму а папа читал газету
Restored: Мама мыла раму, а папа читал газету.

Input:    Однако мы решили не идти в кино потому что пошел дождь
Restored: Однако мы решили не идти в кино, потому что пошел дождь.

Input:    Он сказал Привет как дела
Restored: Он сказал: " Привет, как дела".

Input:    Я говорю ей Что думаешь А она мне Да ничего отстань уже
Restored: Я говорю ей: " Что думаешь?" А она мне: " Да ничего, отстань уже".

Input:    После многих страданий А А Пушкин всё-таки написал свои стишки
Restored: После многих страданий А. А. Пушкин всё-таки написал свои стишки.



In [6]:
from pipeline.aggregate import rebuild_master_excel
rebuild_master_excel()

Rebuilt master table -> /home/temari/god please no diploma/restore_punct/results/master_summary.xlsx
  BERT runs   : 12
  Yandex runs : 0


'/home/temari/god please no diploma/restore_punct/results/master_summary.xlsx'